# Metadata Enrichment with LLMs

pysradb now helps you 'enrich' your metadata frame by leveraging recent advancements in (fast) (S)LLMs. The parsed metadata can be fed to an LLM that is 'instructed' to enrich the metadata by returning 9 ontology-based fields::
`organ`, `tissue`, `anatomical_system`, `cell_type`, `disease`, `sex`, `development_stage`, `assay`, `organism`

There are two approaches in which this is possible:

- **LLMs** ([Requires Ollama](https://ollama.com/download)  - local, no API keys)
- **Embeddings** ([Using sentence-transformers](https://huggingface.co/sentence-transformers) with [BioLORD](https://arxiv.org/abs/2311.16075) embedding model)


In [ ]:
!pip install git+https://github.com/saketkc/pysradb


In [1]:

import pandas as pd

from pysradb import SRAweb
from pysradb.metadata_enrichment import create_metadata_extractor
from pysradb.search import GeoSearch

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

/Users/saket/github/pysradb/pysradb/utils.py:16: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Quick Start 

One line enrichment!

**Prerequisites**: Install Ollama (https://ollama.ai) and pull a model: `ollama pull phi3`

The easiest way to enrich metadata is using the `enrich=True` parameter:

In [3]:
from pysradb.sraweb import SRAweb

db = SRAweb()

df = db.metadata("GSE155673", detailed=True, enrich=True)

cols = [
    "sample_title",
    "sex",
    "guessed_sex",
    "tissue",
    "guessed_tissue",
    "guessed_organ",
    "guessed_anatomical_system",
    "guessed_cell_type",
    "guessed_organ",
    "guessed_tissue",
    "guessed_disease",
]
display(df[cols])

Enriching metadata:   0%|          | 0/24 [00:00<?, ?row/s]

,sample_title,sex,guessed_sex,tissue,guessed_organ,guessed_anatomical_system,guessed_tissue,guessed_cell_type,guessed_organ,guessed_tissue,guessed_disease
0,cov_01_RNA,F,female,<NA>,blood,immune system,peripheral blood,pbmc,blood,peripheral blood,covid-19
1,cov_01_antibody,F,female,<NA>,blood,immune system,peripheral blood,pbmc,blood,peripheral blood,covid-19
2,cov_02_RNA,F,female,<NA>,blood,immune system,peripheral blood,pbmc,blood,peripheral blood,covid-19
3,cov_02_antibody,F,female,<NA>,blood,immune system,peripheral blood,pbmc,blood,peripheral blood,covid-19
4,cov_03_RNA,F,female,<NA>,blood,immune system,peripheral blood,pbmc,blood,peripheral blood,covid-19
5,cov_03_antibody,F,female,<NA>,blood,immune system,peripheral blood,pbmc,blood,peripheral blood,covid-19
6,cov_04_RNA,M,male,<NA>,blood,immune system,peripheral blood,pbmc,blood,peripheral blood,covid-19
7,cov_04_antibody,M,male,<NA>,blood,immune system,peripheral blood,pbmc,blood,peripheral blood,covid-19
8,cov_07_RNA,F,female,<NA>,blood,immune system,peripheral blood,pbmc,blood,peripheral blood,healthy
9,cov_07_antibody,F,female,<NA>,blood,immune system,peripheral blood,pbmc,blood,peripheral blood,healthy


---

## Manual Enrichment


For more control, you can manually create extractors and enrich DataFrames:

### Manual LLM-Based Enrichment


In [5]:
# Create LLM extractor
extractor_llm = create_metadata_extractor(method="llm", backend="ollama/phi3")

# Test extraction
text = (
    "Single-cell RNA-seq of CD4+ T cells from breast cancer patients. sex: F. age: 55"
)
result = extractor_llm.extract_metadata(text)

print("Extracted metadata:")
for key, value in result.items():
    if value != "Unknown":
        print(f"  {key}: {value}")

Extracted metadata:
  organ: breast
  anatomical_system: digestive system
  cell_type: cd4+ t cell
  disease: cancer
  sex: female
  development_stage: adult
  assay: single-cell rna seq
  organism: homo sapiens


In [6]:
if not df.empty:
    df_enriched = extractor_llm.enrich_dataframe(
        df.head(3), text_column="sample_title", prefix="guessed_"
    )

    cols = ["sample_title"] + [
        c for c in df_enriched.columns if c.startswith("guessed_")
    ]
    display(df_enriched[cols])

Enriching metadata:   0%|          | 0/3 [00:00<?, ?row/s]

,sample_title,guessed_organ,guessed_tissue,guessed_anatomical_system,guessed_cell_type,guessed_disease,guessed_sex,guessed_development_stage,guessed_assay,guessed_organism
0,cov_01_RNA,brain,brain tissue,nervous system,neuron,healthy,unknown,adult,rna-seq,homo sapiens
1,cov_01_antibody,unknown,unknown,unknown,pbmc,healthy,unknown,adult,rna-seq,homo sapiens
2,cov_02_RNA,brain,brain tissue,nervous system,neuron,healthy,unknown,adult,rna-seq,homo sapiens


### Manual Embedding-Based Enrichment

Faster than LLMs, works offline. Load comprehensive ontology reference (31K+ terms):

In [7]:
# Load ontology reference (31K+ terms from UBERON, MONDO, CL)
from pysradb.metadata_enrichment import load_ontology_reference

ontology_ref = load_ontology_reference()
print(f"Loaded {sum(len(v) for v in ontology_ref.values()):,} ontology terms")

# Create embedding extractor
extractor_emb = create_metadata_extractor(
    method="embedding",
    model="FremyCompany/BioLORD-2023",
    reference_categories=ontology_ref,
)

text = "scRNA-seq of CD8+ T cells from melanoma patients"
result = extractor_emb.extract_metadata(text)
print(f"\nExtracted: {result}")

Loaded 31,164 ontology terms

Extracted: {'organ': 'Unknown', 'tissue': 'mucosa-associated lymphoid tissue', 'anatomical_system': 'Unknown', 'cell_type': 'mucosal-associated invariant T cell, human', 'disease': 'melanoma, cutaneous malignant, susceptibility to, 8', 'sex': 'Unknown', 'development_stage': 'Unknown', 'assay': 'Unknown', 'organism': 'Unknown'}
